# EDA WhoOwns

Diese Vorlage trennt die Analyse von `data_acquisition` und arbeitet nur mit den bereinigten Dateien aus `data/whoOwns/processed/`.

Ziele dieser EDA-Vorlage:

- Wohneigentumsquote nach Alter und Generation vergleichen
- Baby Boomers und Generation X bei Alter 25 gegenüberstellen
- Eine vorbereitete Stelle fuer die Visualisierung der Zimmerzahl in Eigentumswohnungen/Haushalten schaffen

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Project root with data/ folder not found.')

BASE_PATH = find_project_root(Path.cwd())
WHOOWNS_PROCESSED_PATH = BASE_PATH / 'data' / 'whoOwns' / 'processed' / 'wohneigentumsquote_kanton_2026_clean.csv'
ANFRAGE_MARTINEZ_PROCESSED_PATH = BASE_PATH / 'data' / 'whoOwns' / 'processed' / 'anfrage_martinez_20260414_clean.csv'

whoowns = pd.read_csv(WHOOWNS_PROCESSED_PATH)
martinez = pd.read_csv(ANFRAGE_MARTINEZ_PROCESSED_PATH)

whoowns.columns = whoowns.columns.str.lower().str.strip()
martinez.columns = martinez.columns.str.lower().str.strip()

whoowns.head()
martinez.head()

,alter,bewohnertyp,absolute zahlen,vertrauens-intervall : \n± (in %),anteil in %,vertrauens-\nintervall : \n± (in %-pkte),datenqualität,year
0,15,Mieter oder Genossenschafter,40110.0,6.2,46.0,2.0,NaN,2024
1,15,Eigentümer,41551.0,5.6,47.7,2.0,NaN,2024
2,15,Andere Situation2),5534.0,17.1,6.3,1.0,NaN,2024
3,15,Total,87196.0,4.0,100.0,NaN,NaN,2024
4,16,Mieter oder Genossenschafter,41074.0,6.5,47.0,2.1,NaN,2024


In [2]:
def generation_from_birthyear(year: float) -> str:
    if pd.isna(year):
        return pd.NA
    if year <= 1945:
        return 'Silent Generation'
    if 1946 <= year <= 1964:
        return 'Baby Boomers'
    if 1965 <= year <= 1980:
        return 'Generation X'
    if 1981 <= year <= 1996:
        return 'Millennials'
    return 'Generation Z'

def prepare_martinez(df: pd.DataFrame) -> pd.DataFrame:
    clean = df.copy()
    clean['birthyear'] = clean['year'] - clean['alter']
    clean['generation'] = clean['birthyear'].apply(generation_from_birthyear)
    clean['is_owner'] = clean['bewohnertyp'].eq('Eigentümer')
    clean['is_owner'] = clean['is_owner'].fillna(False)
    return clean

martinez = prepare_martinez(martinez)
martinez[['alter', 'generation', 'bewohnertyp', 'is_owner']].head()

,alter,generation,bewohnertyp,is_owner
0,15,Generation Z,Mieter oder Genossenschafter,False
1,15,Generation Z,Eigentümer,True
2,15,Generation Z,Andere Situation2),False
3,15,Generation Z,Total,False
4,16,Generation Z,Mieter oder Genossenschafter,False


In [3]:
def owner_rate_by_generation_and_age(df: pd.DataFrame, age: int, generations: list[str]) -> pd.DataFrame:
    subset = df[(df['alter'] == age) & (df['generation'].isin(generations))].copy()
    if subset.empty:
        return pd.DataFrame(columns=['generation', 'owner_rate', 'n'])
    summary = (
        subset.groupby('generation', dropna=False)
        .agg(owner_rate=('is_owner', 'mean'), n=('is_owner', 'size'))
        .reindex(generations)
        .reset_index()
    )
    summary['owner_rate'] = summary['owner_rate'] * 100
    return summary

def plot_owner_pies_for_age(df: pd.DataFrame, age: int, generations: list[str]) -> pd.DataFrame:
    summary = owner_rate_by_generation_and_age(df, age, generations)
    if summary.empty:
        print(f'Keine Daten fuer Alter {age} und die gewaehlten Generationen gefunden.')
        return summary

    fig, axes = plt.subplots(1, len(summary), figsize=(6 * len(summary), 5))
    if len(summary) == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, summary.iterrows()):
        owner_rate = float(row['owner_rate']) if pd.notna(row['owner_rate']) else 0.0
        non_owner_rate = max(0.0, 100.0 - owner_rate)
        ax.pie(
            [owner_rate, non_owner_rate],
            labels=['Eigentum', 'Kein Eigentum'],
            autopct='%1.1f%%',
            startangle=90,
            colors=['#1b4332', '#d9d9d9'],
        )
        ax.set_title(f"{row['generation']} | Alter {age}\n(n={int(row['n'])})")
    plt.suptitle('Wohneigentumsquote nach Generation und Alter', y=1.03, fontsize=14)
    plt.tight_layout()
    plt.show()
    return summary

selected_age = 25
selected_generations = ['Baby Boomers', 'Generation X']
summary_age_25 = plot_owner_pies_for_age(martinez, selected_age, selected_generations)
summary_age_25

Keine Daten fuer Alter 25 und die gewaehlten Generationen gefunden.


,generation,owner_rate,n


In [4]:
room_candidates = [
    'zimmer',
    'rooms',
    'anz_zimmer',
    'anzahl_zimmer',
    'wohnungszimmer',
    'room_count',
    'num_rooms',
]

room_column = next((col for col in martinez.columns if col in room_candidates), None)

if room_column is None:
    print('In den aktuellen WhoOwns/Anfrage-Martinez-Daten ist keine Zimmer-Spalte vorhanden.')
    print('Wenn du die Zimmerzahl als zweite Quelle hinzufuegst, kann hier direkt eine Verteilung fuer Eigentumswohnungen gezeichnet werden.')
else:
    owner_only = martinez[martinez['is_owner']].copy()
    plt.figure(figsize=(8, 5))
    owner_only[room_column].dropna().astype(str).value_counts().sort_index().plot(kind='bar', color='#457b9d')
    plt.title('Zimmerzahl der Eigentümer-Haushalte')
    plt.xlabel('Zimmerzahl')
    plt.ylabel('Anzahl')
    plt.tight_layout()
    plt.show()

In den aktuellen WhoOwns/Anfrage-Martinez-Daten ist keine Zimmer-Spalte vorhanden.
Wenn du die Zimmerzahl als zweite Quelle hinzufuegst, kann hier direkt eine Verteilung fuer Eigentumswohnungen gezeichnet werden.
